# NB09 — Multi-Metric Weighted Scoring

**Purpose:** Combine all individual metric CSVs (nb03–nb08) into a composite score.

**Experiments:** Equal-weight, physics-informed, grid-search (in-sample), LOOCV-optimised (honest)

**Runs on:** Local CPU

## 0. Install Dependencies

In [1]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'pydicom', 'numpy', 'pandas', 'matplotlib', 'Pillow', 'openpyxl', 'scipy'])
print('Dependencies OK')


Dependencies OK


In [2]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('.'))

import numpy as np
import pandas as pd
from itertools import product
import matplotlib
matplotlib.rcParams['figure.facecolor'] = '#0f0f1a'
matplotlib.rcParams['axes.facecolor']   = '#1a1a2e'
matplotlib.rcParams['text.color']       = 'white'
%matplotlib inline

GT_PATH = r'E:/CINE-Test1/1stcohortSAX (1).xlsx'
OUT_DIR = 'outputs'
os.makedirs(OUT_DIR, exist_ok=True)
EXCLUDE = ['Patient-KP']
OFFSETS_13 = [-150 + i*25 for i in range(13)]

SIGMA_CENTER = 100.0
def gaussian_center_prior(hz_array):
    g = np.exp(-0.5 * (np.array(hz_array) / SIGMA_CENTER)**2)
    mn, mx = g.min(), g.max()
    return (g - mn) / (mx - mn) if mx > mn else g

print('Config OK')


Config OK


## 1. Load All Metric CSVs

In [3]:
from utils.gt_loader import load_gt

gt_dict = load_gt(GT_PATH)

df_signal   = pd.read_csv(f'{OUT_DIR}/signal_metrics.csv')
df_contrast = pd.read_csv(f'{OUT_DIR}/contrast_metrics.csv')
df_banding  = pd.read_csv(f'{OUT_DIR}/banding_metrics.csv')
df_edge     = pd.read_csv(f'{OUT_DIR}/edge_metrics.csv')
df_temporal = pd.read_csv(f'{OUT_DIR}/temporal_metrics.csv')

cnn_path = f'{OUT_DIR}/cnn_scores.csv'
if os.path.exists(cnn_path):
    df_cnn  = pd.read_csv(cnn_path)
    has_cnn = True
    print('CNN scores loaded.')
else:
    has_cnn = False
    print('WARNING: cnn_scores.csv not found — CNN component excluded. Run nb08 on Kaggle first.')

KEY = ['patient_id', 'image_no']
master = df_signal[KEY + ['offset_hz', 'mu_myo', 'snr_myo', 'homogeneity']].copy()
master = master.merge(df_contrast[KEY + ['cnr', 'contrast_ratio']], on=KEY)
master = master.merge(df_banding[KEY  + ['banded_fraction_myo', 'banding_burden', 'band_trough_depth']], on=KEY)
master = master.merge(df_edge[KEY     + ['edge_sharpness_boundary', 'edge_sharpness_myo', 'laplacian_variance']], on=KEY)
master = master.merge(df_temporal[KEY + ['temporal_std_sweep', 'local_signal_drop']], on=KEY)
if has_cnn:
    master = master.merge(df_cnn[KEY  + ['cnn_score']], on=KEY, how='left')
    master['cnn_score'] = master['cnn_score'].fillna(0.5)
else:
    master['cnn_score'] = 0.5

master = master[~master['patient_id'].isin(EXCLUDE)].copy()
print(f'Master: {len(master)} rows, {master["patient_id"].nunique()} patients')
master.head()


CNN scores loaded.
Master: 324 rows, 25 patients


,patient_id,image_no,offset_hz,mu_myo,snr_myo,homogeneity,cnr,contrast_ratio,banded_fraction_myo,banding_burden,band_trough_depth,edge_sharpness_boundary,edge_sharpness_myo,laplacian_variance,temporal_std_sweep,local_signal_drop,cnn_score
0,Patient-BEV,1,-150.0,0.095261,1.6526,0.6573,1.7351,2.0499,0.08968,0.0,0.51436,0.214429,0.135504,0.001286,0.013323,0.000000,0.002360
1,Patient-BEV,2,-125.0,0.068615,1.0967,0.6844,2.2659,3.0661,0.09337,0.0,0.51010,0.210204,0.112044,0.001186,0.013636,0.011263,0.000161
2,Patient-BEV,3,-100.0,0.064495,1.1093,0.7551,1.7740,2.5992,0.04300,0.0,0.37752,0.144418,0.085448,0.000912,0.002413,0.001254,0.000863
3,Patient-BEV,4,-75.0,0.062884,1.0734,0.8065,1.3636,2.2704,0.01720,0.0,0.28154,0.106979,0.068276,0.000828,0.001513,0.000000,0.004681
4,Patient-BEV,5,-50.0,0.060800,1.0083,0.8190,1.1133,2.1041,0.00983,0.0,0.25729,0.098605,0.063906,0.000778,0.000893,0.001277,0.019496


## 2. Normalise & Composite Functions

In [4]:
def minmax_per_patient(df, col):
    def _norm(x):
        r = x.max() - x.min()
        return (x - x.min()) / r if r > 1e-9 else x * 0 + 0.5
    return df.groupby('patient_id')[col].transform(_norm)

master['n_mu_myo']              = minmax_per_patient(master, 'mu_myo')
master['n_cnr']                 = minmax_per_patient(master, 'cnr')
master['n_banded_fraction_myo'] = minmax_per_patient(master, 'banded_fraction_myo')
master['n_edge_boundary']       = minmax_per_patient(master, 'edge_sharpness_boundary')
master['n_temporal_std']        = minmax_per_patient(master, 'temporal_std_sweep')
master['n_cnn_score']           = minmax_per_patient(master, 'cnn_score')
master['center_prior']          = master.groupby('patient_id')['offset_hz'].transform(
    lambda hz: pd.Series(gaussian_center_prior(hz.values), index=hz.index)
)
print('Normalisation done.')

from utils.gt_loader import evaluate_predictions
from utils.vis_utils import summary_text

def compute_composite(df, w):
    score = (
        w[0] * df['n_mu_myo']
      + w[1] * df['n_cnr']
      + w[2] * (1 - df['n_banded_fraction_myo'])
      + w[3] * df['n_edge_boundary']
      + w[4] * (1 - df['n_temporal_std'])
      + w[5] * df['n_cnn_score']
      + w[6] * df['center_prior']
    )
    df = df.copy()
    df['composite_score'] = score
    return df

def evaluate_composite(df, gt_dict, weights):
    df_s = compute_composite(df, weights)
    pred_dict = {}
    for pid, grp in df_s.groupby('patient_id'):
        best_idx = int(np.argmax(grp.sort_values('image_no')['composite_score'].values))
        pred_dict[pid] = best_idx + 1
    return evaluate_predictions(pred_dict, gt_dict, exclude=EXCLUDE)

print('Composite functions defined.')


Normalisation done.
Composite functions defined.


## 3. Experiments A–D: Equal / Physics / Grid-Search / LOOCV

In [5]:
# A: Equal weight
W_EQUAL = [1/7]*7
res_equal = evaluate_composite(master, gt_dict, W_EQUAL)
print(summary_text('Equal-weight composite (1/7)', res_equal['top1'], res_equal['pm1'], res_equal['mae_hz'], res_equal['n_valid']))

# B: Physics-informed
W_PHYSICS = [0.05, 0.05, 0.55, 0.10, 0.05, 0.10, 0.10]
res_physics = evaluate_composite(master, gt_dict, W_PHYSICS)
print(summary_text('Physics-informed weights', res_physics['top1'], res_physics['pm1'], res_physics['mae_hz'], res_physics['n_valid']))



  Equal-weight composite (1/7)
  Top-1 accuracy : 4/25 = 16.0%
  ±1 step        : 11/25 = 44.0%
  MAE            : 46.0 Hz


  Physics-informed weights
  Top-1 accuracy : 9/25 = 36.0%
  ±1 step        : 18/25 = 72.0%
  MAE            : 28.0 Hz



In [6]:
# C: Random search (in-sample) — ~30 seconds
import numpy as np
np.random.seed(42)

print('Running random weight search (in-sample, 2000 samples)...')
N_SAMPLES   = 2000
best_top1   = 0.0
best_pm1    = 0.0
best_w      = [1/7]*7

for i in range(N_SAMPLES):
    raw = np.random.dirichlet(np.ones(7))   # random weights summing to 1
    res = evaluate_composite(master, gt_dict, raw.tolist())
    if res['top1'] > best_top1 or (res['top1'] == best_top1 and res['pm1'] > best_pm1):
        best_top1, best_pm1, best_w = res['top1'], res['pm1'], raw.tolist()

print(f'Searched {N_SAMPLES} random combinations.')
res_grid = evaluate_composite(master, gt_dict, best_w)
print(summary_text('Random-search (IN-SAMPLE)', res_grid['top1'], res_grid['pm1'], res_grid['mae_hz'], res_grid['n_valid']))

labels = ['signal', 'cnr', 'no_bfm', 'edge', 'no_temp', 'cnn', 'center']
for lbl, w in zip(labels, best_w):
    print(f'  {lbl:15s}: {w:.3f}')


Running random weight search (in-sample, 2000 samples)...
Searched 2000 random combinations.

  Random-search (IN-SAMPLE)
  Top-1 accuracy : 14/25 = 56.0%
  ±1 step        : 21/25 = 84.0%
  MAE            : 17.0 Hz

  signal         : 0.080
  cnr            : 0.015
  no_bfm         : 0.347
  edge           : 0.035
  no_temp        : 0.097
  cnn            : 0.204
  center         : 0.223


In [7]:
# D: LOOCV scipy.optimize — honest, continuous, fast (~5-10 min)
from scipy.optimize import minimize

print('Running LOOCV scipy.optimize weight search (honest)...')
patient_ids = sorted(master['patient_id'].unique())
loocv_preds = {}

def neg_top1(raw_weights, df, gt_dict):
    w = np.abs(raw_weights)
    w = w / (w.sum() + 1e-9)
    res = evaluate_composite(df, gt_dict, w.tolist())
    return -res['top1']

W0     = np.array([1/7]*7)
BOUNDS = [(0.0, 1.0)] * 7

for fold_i, test_pid in enumerate(patient_ids):
    train_df = master[master['patient_id'] != test_pid].copy()
    test_df  = master[master['patient_id'] == test_pid].copy()

    result = minimize(
        neg_top1, W0,
        args=(train_df, gt_dict),
        method='L-BFGS-B',
        bounds=BOUNDS,
        options={'maxiter': 200, 'ftol': 1e-4}
    )
    best_w_fold = np.abs(result.x)
    best_w_fold = (best_w_fold / (best_w_fold.sum() + 1e-9)).tolist()

    test_scored = compute_composite(test_df, best_w_fold)
    best_idx    = int(np.argmax(test_scored.sort_values('image_no')['composite_score'].values))
    loocv_preds[test_pid] = best_idx + 1

    gt_no  = gt_dict[test_pid]['image_no']
    marker = '✅' if best_idx+1 == gt_no else ('~' if abs(best_idx+1 - gt_no) <= 1 else '❌')
    print(f'  [{fold_i+1:2d}/{len(patient_ids)}] {test_pid}: pred=#{best_idx+1}  gt=#{gt_no}  {marker}')

res_loocv = evaluate_predictions(loocv_preds, gt_dict, exclude=EXCLUDE)
print(summary_text('LOOCV scipy.optimize (HONEST)', res_loocv['top1'], res_loocv['pm1'], res_loocv['mae_hz'], res_loocv['n_valid']))


Running LOOCV scipy.optimize weight search (honest)...
  [ 1/25] Patient-BEV: pred=#8  gt=#7  ~
  [ 2/25] Patient-BGT: pred=#7  gt=#7  ✅
  [ 3/25] Patient-BVE: pred=#7  gt=#6  ~
  [ 4/25] Patient-CEA: pred=#7  gt=#7  ✅
  [ 5/25] Patient-CEJP: pred=#2  gt=#4  ❌
  [ 6/25] Patient-DEM: pred=#11  gt=#9  ❌
  [ 7/25] Patient-FAJ: pred=#4  gt=#9  ❌
  [ 8/25] Patient-FEC: pred=#6  gt=#8  ❌
  [ 9/25] Patient-GJ: pred=#9  gt=#9  ✅
  [10/25] Patient-HMJ: pred=#8  gt=#8  ✅
  [11/25] Patient-HRJ: pred=#5  gt=#7  ❌
  [12/25] Patient-IAM: pred=#10  gt=#8  ❌
  [13/25] Patient-INA: pred=#6  gt=#11  ❌
  [14/25] Patient-MJ: pred=#4  gt=#7  ❌
  [15/25] Patient-MS: pred=#8  gt=#9  ~
  [16/25] Patient-MSR: pred=#8  gt=#10  ❌
  [17/25] Patient-OAF: pred=#6  gt=#9  ❌
  [18/25] Patient-PJ: pred=#5  gt=#8  ❌
  [19/25] Patient-RAK: pred=#8  gt=#7  ~
  [20/25] Patient-RK: pred=#6  gt=#9  ❌
  [21/25] Patient-SK: pred=#11  gt=#8  ❌
  [22/25] Patient-SWJ: pred=#8  gt=#7  ~
  [23/25] Patient-TL: pred=#8  gt=#9  ~
  [

## 4. Save Composite CSVs for nb10

In [8]:
for w_name, w_vals in [('physics', W_PHYSICS), ('gridsearch', best_w), ('equal', W_EQUAL)]:
    df_out = compute_composite(master, w_vals)
    df_out[['patient_id','image_no','offset_hz','composite_score']].to_csv(
        f'{OUT_DIR}/composite_{w_name}.csv', index=False)
    print(f'Saved composite_{w_name}.csv')

print('NB09 complete. Proceed to nb10_evaluation.ipynb')

Saved composite_physics.csv
Saved composite_gridsearch.csv
Saved composite_equal.csv
NB09 complete. Proceed to nb10_evaluation.ipynb


In [9]:
from scipy.optimize import minimize
import numpy as np

print('Running LOOCV scipy.optimize weight search (honest)...')
print('This finds the best weights on training patients and tests on the held-out patient.\n')

patient_ids = sorted(master['patient_id'].unique())
loocv_preds = {}
loocv_best_weights = {}

def neg_top1(raw_weights, df, gt_dict):
    """Objective: maximise Top-1 accuracy (minimise negative)."""
    w = np.abs(raw_weights)
    w = w / (w.sum() + 1e-9)
    res = evaluate_composite(df, gt_dict, w.tolist())
    return -res['top1']

W0     = np.array([1/7]*7)     # start: equal weights
BOUNDS = [(0.0, 1.0)] * 7     # each weight in [0, 1]

for fold_i, test_pid in enumerate(patient_ids):
    train_df = master[master['patient_id'] != test_pid].copy()
    test_df  = master[master['patient_id'] == test_pid].copy()

    # Optimise weights on training patients only
    result = minimize(
        neg_top1,
        W0,
        args=(train_df, gt_dict),
        method='L-BFGS-B',
        bounds=BOUNDS,
        options={'maxiter': 300, 'ftol': 1e-5}
    )

    best_w_fold = np.abs(result.x)
    best_w_fold = (best_w_fold / (best_w_fold.sum() + 1e-9)).tolist()
    loocv_best_weights[test_pid] = best_w_fold

    # Predict on the held-out test patient using training-optimised weights
    test_scored = compute_composite(test_df, best_w_fold)
    best_idx    = int(np.argmax(test_scored.sort_values('image_no')['composite_score'].values))
    loocv_preds[test_pid] = best_idx + 1

    gt_no  = gt_dict[test_pid]['image_no']
    diff   = abs(best_idx + 1 - gt_no)
    marker = 'OK' if diff == 0 else ('~+/-1' if diff == 1 else 'FAIL')
    print(f'  [{fold_i+1:2d}/{len(patient_ids)}] {test_pid:15s}: pred=#{best_idx+1:2d}  gt=#{gt_no:2d}  {marker}')

# Final evaluation
res_loocv = evaluate_predictions(loocv_preds, gt_dict, exclude=EXCLUDE)
print()
print(summary_text('LOOCV scipy.optimize (HONEST)', res_loocv['top1'], res_loocv['pm1'], res_loocv['mae_hz'], res_loocv['n_valid']))

# Show average weights learned across folds
import pandas as pd
w_df = pd.DataFrame(loocv_best_weights, index=['signal','cnr','no_bfm','edge','no_temp','cnn','center']).T
print('\nMean weights across all folds:')
print(w_df.mean().round(3).to_string())

# Save LOOCV predictions
loocv_df = pd.DataFrame([
    {'patient_id': pid, 'predicted_image_no': pred_no, 'gt_image_no': gt_dict[pid]['image_no']}
    for pid, pred_no in loocv_preds.items()
])
loocv_df.to_csv(f'{OUT_DIR}/loocv_predictions.csv', index=False)
print(f'\nSaved loocv_predictions.csv')


Running LOOCV scipy.optimize weight search (honest)...
This finds the best weights on training patients and tests on the held-out patient.

  [ 1/25] Patient-BEV    : pred=# 8  gt=# 7  ~+/-1
  [ 2/25] Patient-BGT    : pred=# 7  gt=# 7  OK
  [ 3/25] Patient-BVE    : pred=# 7  gt=# 6  ~+/-1
  [ 4/25] Patient-CEA    : pred=# 7  gt=# 7  OK
  [ 5/25] Patient-CEJP   : pred=# 2  gt=# 4  FAIL
  [ 6/25] Patient-DEM    : pred=#11  gt=# 9  FAIL
  [ 7/25] Patient-FAJ    : pred=# 4  gt=# 9  FAIL
  [ 8/25] Patient-FEC    : pred=# 6  gt=# 8  FAIL
  [ 9/25] Patient-GJ     : pred=# 9  gt=# 9  OK
  [10/25] Patient-HMJ    : pred=# 8  gt=# 8  OK
  [11/25] Patient-HRJ    : pred=# 5  gt=# 7  FAIL
  [12/25] Patient-IAM    : pred=#10  gt=# 8  FAIL
  [13/25] Patient-INA    : pred=# 6  gt=#11  FAIL
  [14/25] Patient-MJ     : pred=# 4  gt=# 7  FAIL
  [15/25] Patient-MS     : pred=# 8  gt=# 9  ~+/-1
  [16/25] Patient-MSR    : pred=# 8  gt=#10  FAIL
  [17/25] Patient-OAF    : pred=# 6  gt=# 9  FAIL
  [18/25] Patie